# Function-by-function R ⇄ Python dictionary

For R users porting Pando code: every public function, its parameters, the R call, the Python call on the same input, and the numerical comparison. Per-function R outputs come from `examples/r_per_function_dump.R` (one deterministic design data frame, small ranges, matrices); pipeline functions are compared on the canonical fixture (`tests/r_reference_driver.R`).

In [1]:
import json, os, sys
from pathlib import Path
import numpy as np, pandas as pd, yaml
import matplotlib.pyplot as plt
from scipy import stats
HERE = Path.cwd()
sys.path.insert(0, str(HERE.parent / "tests"))
import parity_check as pc
WORK = Path(os.environ["PYPANDO_WORK"])          # holds ref/, cand_final/, ref_xgbdiag/, ref_fn/, bench/, probe/
REF, CAND, XGBDIAG = WORK / "ref", WORK / "cand_final", WORK / "ref_xgbdiag"
manifest = yaml.safe_load((HERE.parent / "data" / "manifest.yaml").read_text())
import pypando as pp
from pypando.formula import PandoFormula
from pypando.rrng import RRandom
FN = WORK / "ref_fn"
d = pd.read_csv(FN / "design.csv"); data = {k: d[k].to_numpy() for k in d.columns}
frm = PandoFormula.from_string("GENE ~ chr1_100_600:TFA + chr1_100_600:TFB + chr1_900_1400:TFA + chr2_50_550:NKX2_1 + chr2_50_550:TFB")
verdicts = []
def compare(name, res, cols, key="term"):
    r = pd.read_csv(FN / f"{name}_coefs.csv"); c = res["coefs"]
    m = r.merge(c, on=key, suffixes=("_r", "_c"))
    err = {col: float(np.nanmax(np.abs(m[col + "_r"] - m[col + "_c"]) / np.maximum(np.abs(m[col + "_r"]), 1e-300))) for col in cols}
    drsq = float(pd.read_csv(FN / f"{name}_gof.csv")["rsq"].iloc[0] - res["gof"]["rsq"])
    ok = list(r[key]) == list(c[key]) and max(err.values()) < 1e-10 and abs(drsq) < 1e-12
    verdicts.append({"function": name, "output": "coefs + rsq", "metric": "max rel. diff", "value": max(err.values()), "pass": ok})
    return pd.DataFrame({"max relative difference": err, "terms identical": list(r[key]) == list(c[key]), "rsq difference": drsq})

## `fit_glm`

Gaussian generalised linear model and its coefficient table (`stats::glm` + `summary`).

| R name | Python name | Type | Default | Range / values | Description |
|---|---|---|---|---|---|
| formula | formula | PandoFormula | — | Pando-style terms | model formula (`PandoFormula.from_string` parses R's form) |
| data | data | dict[str, array] | — | columns by name | model variables (R: data.frame) |
| family | family | str | `gaussian` | `'gaussian'` | only the Gaussian family is ported (Pando's default) |
| ... | — | — | — | — | passed to `glm()` in R; not used |

**R**
```r
fit_glm(frm, d)
```

In [2]:
res = pp.fit_glm(frm, data)
compare('fit_glm', res, ('estimate', 'std_err', 'statistic', 'pval'))

,max relative difference,terms identical,rsq difference
estimate,5.957230e-15,True,4.440892e-16
std_err,2.492317e-15,True,4.440892e-16
statistic,5.376053e-15,True,4.440892e-16
pval,6.259592e-14,True,4.440892e-16


## `fit_glmnet`

Elastic net path; coefficients at the first lambda with dev.ratio > 0.95 x max.

| R name | Python name | Type | Default | Range / values | Description |
|---|---|---|---|---|---|
| formula | formula | PandoFormula | — |  | model formula |
| data | data | dict | — |  | model variables |
| family | family | str | `gaussian` (R: the *function*) | `'gaussian'` | py-Pando ports the character family (how `infer_grn` calls it); R's function default runs glmnet's `glmnet.path` solver, not ported |
| alpha | alpha | float | 0.5 | [0, 1] | elastic-net mixing |
| ... | — | — | — | — | glmnet options; not exposed |

**R**
```r
fit_glmnet(frm, d, family = 'gaussian', alpha = 0.5)
```

In [3]:
res = pp.fit_glmnet(frm, data, alpha=0.5)
compare('fit_glmnet', res, ('estimate',))

,max relative difference,terms identical,rsq difference
estimate,1.224802e-14,True,1.110223e-16


## `fit_cvglmnet`

10-fold cross-validated elastic net; coefficients at lambda.1se.

| R name | Python name | Type | Default | Range / values | Description |
|---|---|---|---|---|---|
| formula | formula | PandoFormula | — |  | model formula |
| data | data | dict | — |  | model variables |
| family | family | str | `gaussian` | `'gaussian'` | as fit_glmnet |
| alpha | alpha | float | 0.5 | [0, 1] | elastic-net mixing |
| — | rng | RRandom | None |  | **new in Python**: R's random stream; `RRandom(42)` = `set.seed(42)` |
| ... | foldid, nfolds, fold_lambda |  | None, 10, 'own' |  | fold control; `fold_lambda='own'` = glmnet 5.0 behaviour |

**R**
```r
set.seed(42); fit_cvglmnet(frm, d, family = 'gaussian', alpha = 0.5)
```

In [4]:
res = pp.fit_cvglmnet(frm, data, alpha=0.5, rng=RRandom(42))
compare('fit_cvglmnet', res, ('estimate',))

,max relative difference,terms identical,rsq difference
estimate,1.963540e-14,True,1.110223e-16


## `fit_xgb_params_honoured`

Gradient boosting (xgboost) and gain / cover / frequency importance. Compared with Pando's parameters honoured (see notebook 1).

| R name | Python name | Type | Default | Range / values | Description |
|---|---|---|---|---|---|
| formula | formula | PandoFormula | — |  | model formula |
| data | data | dict | — |  | model variables |
| params | params | dict | max_depth=3, eta=0.01, reg:squarederror |  | xgboost parameters (dropped by R xgboost >= 3) |
| nrounds | nrounds | int | 1000 | > 0 | boosting rounds |
| nthread | nthread | int | 1 |  | threads |

**R**
```r
xgb.train(params = list(max_depth = 3, eta = 0.01), ...)
```

In [5]:
res = pp.fit_xgb(frm, data)
compare('fit_xgb_params_honoured', res, ('gain', 'cover', 'frequency'))

,max relative difference,terms identical,rsq difference
gain,9.118388e-16,True,0.0
cover,1.915605e-15,True,0.0
frequency,2.281705e-15,True,0.0


## `fit_bagging_ridge`

Bagging of 200 ridge regressions (scikit-learn), Wilcoxon / t p-values over bags.

| R name | Python name | Type | Default | Range / values | Description |
|---|---|---|---|---|---|
| formula | formula | PandoFormula | — |  | model formula |
| data | data | dict | — |  | model variables |
| alpha | alpha | float | 1 (0.5 from infer_grn) | > 0 | ridge penalty |
| solver | solver | str | 'auto' | sklearn solvers | ridge solver |
| bagging_number | bagging_number | int | 200 | > 0 | number of bags |
| n_jobs | n_jobs | int | 1 |  | sklearn jobs |
| p_method | p_method | str | 'wilcox' | 'wilcox', 't' | test on the bag coefficients |

**R**
```r
np$random$seed(42L); fit_bagging_ridge(frm, d, alpha = 0.5)
```

In [6]:
np.random.seed(42)
res = pp.fit_bagging_ridge(frm, data, alpha=0.5)
compare('fit_bagging_ridge', res, ('estimate', 'pval'))

,max relative difference,terms identical,rsq difference
estimate,4.158334e-15,True,3.330669e-16
pval,9.721460e-15,True,3.330669e-16


## `fit_bayesian_ridge`

Bayesian ridge (scikit-learn) with normal-approximation p-values.

| R name | Python name | Type | Default | Range / values | Description |
|---|---|---|---|---|---|
| formula | formula | PandoFormula | — |  | model formula |
| data | data | dict | — |  | model variables |
| ... | **kwargs |  |  |  | passed to `BayesianRidge` |

**R**
```r
fit_bayesian_ridge(frm, d)
```

In [7]:
res = pp.fit_bayesian_ridge(frm, data)
compare('fit_bayesian_ridge', res, ('estimate', 'est_variance', 'pval'))

,max relative difference,terms identical,rsq difference
estimate,5.584347e-14,True,-3.330669e-16
est_variance,4.169887e-14,True,-3.330669e-16
pval,3.725388e-13,True,-3.330669e-16


## `fit_model`

Dispatch to the `fit_*` function for `method`.

| R name | Python name | Type | Default | Range / values | Description |
|---|---|---|---|---|---|
| formula | formula | PandoFormula | — |  | model formula |
| data | data | dict | — |  | model variables |
| method | method | str | 'glm' | glm, glmnet, cv.glmnet, brms, xgb, bagging_ridge, bayesian_ridge | model (brms not ported) |
| family | family | str | gaussian |  | error family |
| alpha | alpha | float | 1 |  | elastic net / ridge parameter |

**R**
```r
fit_model(frm, d, method = 'glm')
```

In [8]:
res = pp.fit_model(frm, data, method='glm')
compare('fit_model', res, ('estimate', 'std_err', 'pval'))

,max relative difference,terms identical,rsq difference
estimate,5.957230e-15,True,4.440892e-16
std_err,2.492317e-15,True,4.440892e-16
pval,6.259592e-14,True,4.440892e-16


## `format_coefs`

Split model terms into TF and region, add BH-adjusted p-values.

| R name | Python name | Type | Default | Range / values | Description |
|---|---|---|---|---|---|
| coefs | coefs | DataFrame | — |  | target, term, estimate, ... |
| term | term | str | ':' | ':', '*', '+' | interaction operator |
| adjust_method | adjust_method | str | 'fdr' | p.adjust methods | p-value adjustment |

**R**
```r
format_coefs(cbind(target = 'GENE', fit_glm(frm, d)$coefs), term = ':')
```

In [9]:
tab = pp.fit_glm(frm, data)["coefs"].assign(target="GENE")[["target", "term", "estimate", "std_err", "statistic", "pval"]]
mine = pp.format_coefs(tab); theirs = pd.read_csv(FN / "format_coefs.csv")
same = (mine[["tf", "target", "region", "term"]].values == theirs[["tf", "target", "region", "term"]].values).all()
err = float(np.max(np.abs(mine.padj - theirs.padj) / theirs.padj))
verdicts.append({"function": "format_coefs", "output": "tf/region/padj", "metric": "max rel. diff (padj)", "value": err, "pass": bool(same) and err < 1e-12})
mine

,tf,target,region,term,estimate,std_err,statistic,pval,padj
0,TFA,GENE,chr1-100-600,chr1_100_600:TFA,0.816017,0.036962,22.077151,2.310079e-64,1.386047e-63
1,TFB,GENE,chr1-100-600,chr1_100_600:TFB,-0.035518,0.051834,-0.685226,4.937412e-01,7.406118e-01
2,TFA,GENE,chr1-900-1400,TFA:chr1_900_1400,-0.018054,0.024284,-0.743446,4.578057e-01,7.406118e-01
3,NKX2-1,GENE,chr2-50-550,chr2_50_550:NKX2_1,-0.486572,0.038993,-12.478387,5.766741e-29,1.730022e-28
4,TFB,GENE,chr2-50-550,TFB:chr2_50_550,0.020125,0.045231,0.444939,6.566913e-01,7.631703e-01


## `sparse_cor`

Pearson correlation via Pando's moment formula.

| R name | Python name | Type | Default | Range / values | Description |
|---|---|---|---|---|---|
| x | x | matrix | — | cells x k |  |
| y | y | matrix | NULL | cells x m | NULL: cor(x) |
| method | — |  | 'pearson' |  | only Pearson ported (Pando's default) |
| allow_neg | allow_neg | bool | TRUE |  | keep negative correlations |
| remove_na | remove_na | bool | TRUE |  | NaN -> 0 |
| remove_inf | remove_inf | bool | TRUE |  | Inf -> 1 |

**R**
```r
sparse_cor(X, Y); sparse_cor(X)
```

In [10]:
X = d[["TFA", "TFB", "NKX2_1"]].to_numpy(); Y = d[["GENE"]].to_numpy()
e1 = np.max(np.abs(pp.sparse_cor(X, Y) - pd.read_csv(FN / "sparse_cor.csv").to_numpy()))
e2 = np.max(np.abs(pp.sparse_cor(X) - pd.read_csv(FN / "sparse_cor_self.csv").to_numpy()))
verdicts.append({"function": "sparse_cor", "output": "cor", "metric": "max abs diff", "value": float(max(e1, e2)), "pass": max(e1, e2) < 1e-12})
e1, e2

(np.float64(9.992007221626409e-16), np.float64(1.4016565685892601e-15))

## `aggregate_matrix`

Summarise rows by group.

| R name | Python name | Type | Default | Range / values | Description |
|---|---|---|---|---|---|
| x | x | matrix | — | cells x features |  |
| groups | groups | array | NULL |  | group per row |
| fun | fun | str | 'mean' | mean, sum, count, max, min, median, sd, any, all | summary |

**R**
```r
aggregate_matrix(as.matrix(d[, 1:6]), groups = groups, fun = 'mean')
```

In [11]:
groups = (FN / "groups.txt").read_text().split()
for f in ("mean", "sum", "max", "count"):
    mine = pp.aggregate_matrix(d.iloc[:, :6].to_numpy(), groups, fun=f)
    theirs = pd.read_csv(FN / f"aggregate_matrix_{f}.csv").set_index("group")
    e = float(np.max(np.abs(mine.to_numpy() - theirs.to_numpy())))
    verdicts.append({"function": "aggregate_matrix", "output": f, "metric": "max abs diff", "value": e, "pass": e < 1e-12 and list(mine.index) == list(theirs.index)})
pd.DataFrame(verdicts[-4:])

,function,output,metric,value,pass
0,aggregate_matrix,mean,max abs diff,4.884981e-15,True
1,aggregate_matrix,sum,max abs diff,4.831691e-13,True
2,aggregate_matrix,max,max abs diff,0.000000e+00,True
3,aggregate_matrix,count,max abs diff,0.000000e+00,True


## `find_peaks_near_genes`

| R name | Python name | Type | Default | Range / values | Description |
|---|---|---|---|---|---|
| peaks | peaks | ranges DataFrame | — |  | regions |
| genes | genes | ranges DataFrame | — |  | annotation rows with gene_name |
| sep | sep | tuple | c('-', '-') |  | region string separators |
| method | method | str | 'Signac' | 'Signac', 'GREAT' | assignment rule |
| upstream | upstream | int | 100000 |  | bp upstream |
| downstream | downstream | int | 0 |  | bp downstream |
| extend | extend | int | 1000000 |  | GREAT extension |
| only_tss | only_tss | bool | FALSE |  | window from the TSS only |
| verbose | — |  | TRUE |  | not used |
| — | great_annotation | DataFrame | None |  | **new in Python**: override the bundled Ensembl v93 annotation |

**R**
```r
Pando:::find_peaks_near_genes(peaks, genes, method = 'Signac', upstream = 1e5, downstream = 1000, only_tss = FALSE)
```

In [12]:
from pypando import ranges as R
peaks = R.string_to_ranges((FN / "peaks.txt").read_text().split())
genes = pd.read_csv(FN / "genes.csv")
for only in (False, True):
    h, _, _ = pp.find_peaks_near_genes(peaks, genes, upstream=100000, downstream=1000, only_tss=only)
    c = h.tocoo(); mine = set(zip(c.row + 1, c.col + 1))
    t = pd.read_csv(FN / f"find_peaks_near_genes_tss{str(only).upper()}.csv"); theirs = set(zip(t.i, t.j))
    verdicts.append({"function": "find_peaks_near_genes", "output": f"only_tss={only}", "metric": "identical hits", "value": float(mine == theirs), "pass": mine == theirs})
pd.DataFrame(verdicts[-2:])

,function,output,metric,value,pass
0,find_peaks_near_genes,only_tss=False,identical hits,1.0,True
1,find_peaks_near_genes,only_tss=True,identical hits,1.0,True


## Pipeline functions on the canonical fixture

`initiate_grn`, `find_motifs`, `infer_grn` (every method), `find_modules` and the accessors are compared on the fixture in `compare_R_vs_Python.ipynb`; their parameters:

### `initiate_grn`
| R name | Python name | Type | Default | Range / values | Description |
|---|---|---|---|---|---|
| object | rna, atac, annotation | AnnData x2 + DataFrame | — |  | Seurat object -> RNA / ATAC AnnData + annotation |
| regions | regions | ranges DataFrame | NULL |  | restrict candidates (e.g. phastCons) |
| peak_assay | — |  | 'peaks' |  | the ATAC AnnData |
| rna_assay | — |  | 'RNA' |  | the RNA AnnData |
| exclude_exons | exclude_exons | bool | TRUE |  | subtract exons |
| — | variable_features, rna_layer, peak_layer |  | None |  | **new in Python**: VariableFeatures and layers |

### `find_motifs`
| R name | Python name | Type | Default | Range / values | Description |
|---|---|---|---|---|---|
| object | object | GRNData | — |  |  |
| pfm | pfm | list[PFM] | (required in R) |  | default: Pando's bundled motifs |
| genome | genome | FASTA path / pyfaidx.Fasta | — |  | BSgenome -> FASTA |
| motif_tfs | motif_tfs | DataFrame | NULL (bundled motif2tf) |  | motif-to-TF map |
| bg | bg | str / 4 floats | 'subject' | subject, genome, even, freqs | background |
| p_cutoff | p_cutoff | float | 5e-05 |  | MOODS p-value |
| verbose | verbose | bool | TRUE |  |  |

### `infer_grn` / `fit_grn_models`
| R name | Python name | Type | Default | Range / values | Description |
|---|---|---|---|---|---|
| genes | genes | list | NULL (VariableFeatures) |  | targets |
| network_name | network_name | str | '<method>_network' |  |  |
| peak_to_gene_method | peak_to_gene_method | str | 'Signac' | Signac, GREAT |  |
| upstream / downstream / extend / only_tss | same | int / bool | 1e5 / 0 / 1e6 / FALSE |  |  |
| peak_to_gene_domains | peak_to_gene_domains | ranges | NULL |  | fixed domains |
| parallel | parallel | bool | FALSE |  | not used (sequential) |
| tf_cor | tf_cor | float | 0.1 |  | |cor(TF, gene)| threshold |
| peak_cor | peak_cor | float | 0 |  | |cor(peak, gene)| threshold |
| aggregate_rna_col / aggregate_peaks_col | same |  | NULL |  | not supported (errors in R on a GRNData); aggregate beforehand |
| method | method | str | 'glm' | glm, glmnet, cv.glmnet, xgb, bagging_ridge, bayesian_ridge | brms not ported |
| alpha / family / interaction_term / adjust_method / scale | same |  | 0.5 / 'gaussian' / ':' / 'fdr' / FALSE |  |  |
| — | seed | int | None |  | **new in Python**: R folds for cv.glmnet, NumPy seed for bagging_ridge |
| — | blas_threads | int | 1 |  | **new in Python**: BLAS threads during fitting |

### `find_modules`
| R name | Python name | Type | Default | Range / values | Description |
|---|---|---|---|---|---|
| p_thresh | p_thresh | float | 0.05 |  | padj threshold |
| rsq_thresh | rsq_thresh | float | 0.1 |  | model R^2 |
| nvar_thresh | nvar_thresh | int | 10 |  | min variables |
| min_genes_per_module | min_genes_per_module | int | 5 |  | module size |
| xgb_method / xgb_top | same (Network only) |  | 'tf' / 50 |  | xgb networks |

In [13]:
res = pc.main(REF, CAND)
for k in ("candidate_regions", "motif_matrix", "glm_coefficients", "glm_pvalues", "modules", "glmnet_coefficients",
          "cvglmnet_coefficients", "xgb_gain", "bagging_ridge", "bayesian_ridge"):
    verdicts.append({"function": "pipeline (fixture)", "output": k, "metric": "pre-registered gate", "value": np.nan, "pass": res[k]["pass"]})

PASS  candidate_regions: {"identical": true, "n_ref": 8127, "n_cand": 8127}
PASS  candidate_regions_phastcons: {"identical": true, "n_ref": 7130, "n_cand": 7130}
PASS  peaks_near_genes_signac: {"identical": true, "n_ref": 198776, "n_cand": 198776}
PASS  peaks_near_genes_great: {"identical": true, "n_ref": 96699, "n_cand": 96699}
PASS  motif_matrix: {"agreement": 1.0, "jaccard": 1.0, "n_diff": 0, "n_ref": 1240962, "n_cand": 1240962, "tfs_use_identical": true}
PASS  glm_keys: {"identical": true, "n_ref": 24642, "n_cand": 24642}
PASS  glm_coefficients: {"max_normalised_error": {"estimate": 9.791269464166968e-07, "std_err": 7.62226839598076e-07, "statistic": 9.195519709012728e-07, "corr": 7.835080265876213e-06, "gof_rsq": 4.6011063320287575e-07, "gof_nvariables": 0.0}}
PASS  glm_pvalues: {"max_abs_dlog10p": 4.014566457044566e-13, "max_abs_dlog10padj": 3.9479530755670567e-13, "spearman": 0.9999999999987971, "nan_pattern_identical": true}
PASS  modules: {"mean_jaccard": 1.0, "tf_set_jaccard"

## Aggregate verdict

In [14]:
pd.DataFrame(verdicts)

,function,output,metric,value,pass
0,fit_glm,coefs + rsq,max rel. diff,6.259592e-14,True
1,fit_glmnet,coefs + rsq,max rel. diff,1.224802e-14,True
2,fit_cvglmnet,coefs + rsq,max rel. diff,1.963540e-14,True
3,fit_xgb_params_honoured,coefs + rsq,max rel. diff,2.281705e-15,True
4,fit_bagging_ridge,coefs + rsq,max rel. diff,9.721460e-15,True
5,fit_bayesian_ridge,coefs + rsq,max rel. diff,3.725388e-13,True
6,fit_model,coefs + rsq,max rel. diff,6.259592e-14,True
7,format_coefs,tf/region/padj,max rel. diff (padj),6.155914e-14,True
8,sparse_cor,cor,max abs diff,1.401657e-15,True
9,aggregate_matrix,mean,max abs diff,4.884981e-15,True
